# Bài học 11 - Giao thức Ngữ cảnh Mô hình (MCP)

The **Giao thức Ngữ cảnh Mô hình (MCP)** là một tiêu chuẩn mở cho phép các tác nhân khám phá và sử dụng động các công cụ, tài nguyên, và nguồn dữ liệu khi chạy. Thay vì mã hóa cứng các công cụ vào một tác nhân, MCP cho phép các tác nhân kết nối với các máy chủ bên ngoài công bố khả năng theo yêu cầu.

In this lesson, you'll learn:
- MCP là gì và tại sao nó quan trọng đối với hệ thống tác nhân
- Cách kiến trúc khách-chủ (client-server) của MCP hoạt động
- Cách xây dựng các tác nhân sử dụng cơ chế khám phá công cụ theo kiểu MCP


## Thiết lập

**Yêu cầu tiên quyết:**
- Dự án Azure AI Foundry với mô hình đã triển khai
- Chạy `az login` để xác thực


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Giao thức Ngữ cảnh Mô hình (MCP) là gì?

MCP định nghĩa một cách tiêu chuẩn để các tác nhân AI khám phá và tương tác với các công cụ và nguồn dữ liệu bên ngoài:

- **MCP Server**: Cung cấp các công cụ, tài nguyên và lời nhắc thông qua một giao thức tiêu chuẩn
- **MCP Client**: Phần runtime của tác nhân kết nối tới các máy chủ và khám phá các khả năng có sẵn
- **Dynamic Discovery**: Các tác nhân không cần các công cụ được mã hóa cứng — chúng khám phá những gì có sẵn khi chạy

This is powerful for building extensible agent systems where new capabilities can be added without modifying the agent code.


## Cách MCP hoạt động

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Tác nhân (khách hàng MCP) kết nối tới máy chủ MCP
2. Máy chủ phản hồi bằng một danh sách các công cụ có sẵn và các sơ đồ của chúng
3. Tác nhân sau đó có thể gọi bất kỳ công cụ nào được phát hiện trong quá trình suy luận của nó
4. Kết quả được truyền trở lại thông qua cùng một giao thức


## Mô phỏng Phát hiện Công cụ MCP

Vì một máy chủ MCP thực tế yêu cầu có một tiến trình máy chủ đang chạy, chúng tôi sẽ minh họa mẫu này bằng cách sử dụng các hàm `@tool` mô phỏng những gì một dịch vụ lưu trú được kết nối với MCP sẽ cung cấp.

Trong môi trường sản xuất, các công cụ này sẽ được phát hiện động từ một máy chủ MCP thay vì được định nghĩa cục bộ.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Xây dựng một tác nhân với các công cụ theo phong cách MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP trong môi trường sản xuất

Trong một môi trường sản xuất, MCP cho phép các mô hình mạnh mẽ:

- **Khám phá công cụ động**: Các agent kết nối tới các máy chủ MCP và khám phá công cụ khi chạy
- **Kiến trúc tách rời**: Các nhà cung cấp công cụ có thể cập nhật độc lập với agent
- **Chia sẻ giữa các tổ chức**: Các nhóm có thể công khai khả năng thông qua các máy chủ MCP mà bất kỳ agent nào cũng có thể sử dụng
- **Hỗ trợ Microsoft Agent Framework**: MAF có hỗ trợ client MCP tích hợp sẵn thông qua tích hợp `mcp`

Để sử dụng một máy chủ MCP thực sự với MAF, bạn sẽ kết nối qua `hosted_mcp_tool()` hoặc tích hợp client MCP.

**Tìm hiểu thêm:**
- [Đặc tả MCP](https://modelcontextprotocol.io/)
- [Hỗ trợ MCP của Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Tóm tắt

Trong bài học này, bạn đã học được:
- **MCP** là một tiêu chuẩn mở để khám phá công cụ động giữa các tác nhân và nhà cung cấp công cụ
- **Kiến trúc máy khách-má chủ** cho phép các tác nhân khám phá các khả năng tại thời gian chạy
- MCP cho phép **các hệ thống tác nhân mở rộng, tách rời** nơi các công cụ có thể được thêm vào mà không cần thay đổi mã
- Microsoft Agent Framework cung cấp **hỗ trợ MCP tích hợp sẵn** cho sử dụng trong môi trường sản xuất


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Miễn trừ trách nhiệm:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI Co-op Translator (https://github.com/Azure/co-op-translator). Mặc dù chúng tôi nỗ lực để đảm bảo độ chính xác, xin lưu ý rằng các bản dịch tự động có thể chứa lỗi hoặc không chính xác. Tài liệu gốc bằng ngôn ngữ ban đầu nên được coi là nguồn thông tin có thẩm quyền. Đối với những thông tin quan trọng, nên sử dụng bản dịch do người dịch chuyên nghiệp thực hiện. Chúng tôi không chịu trách nhiệm đối với bất kỳ sự hiểu nhầm hoặc diễn giải sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
